In [ ]:
# CELL 1 — Mount Google Drive
# Purpose: give Colab access to your drive so we can load data + save models.
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")


In [ ]:
# CELL 2 — Set paths and extract dataset if zipped
# Purpose: confirm the ZIP exists, extract it, and set BASE variable.

import zipfile
import os

# EDIT THIS if you put the zip elsewhere
ZIP_PATH = "/content/drive/MyDrive/mel_3ch.zip"
BASE = "/content/processed/mel_3ch"  # expected folder after extraction

if os.path.exists(ZIP_PATH):
    print("✅ Found zip:", ZIP_PATH)
    # Extract only if BASE doesn't already exist to save time
    if not os.path.exists(BASE):
        print("📦 Extracting zip to /content (may take a minute)...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall('/content')
        print("✅ Extraction complete.")
    else:
        print("⚠️ Base directory already exists:", BASE)
else:
    print("❌ ZIP not found at:", ZIP_PATH)
    print("Please upload mel_3ch.zip to the path above or change ZIP_PATH.")
    # List drive root to help find files
    !ls -lh /content/drive/MyDrive/

# Final sanity
if not os.path.exists(BASE):
    raise FileNotFoundError(f"Expected dataset folder not found at {BASE}. Update ZIP_PATH or BASE.")
else:
    print("Using dataset base:", BASE)


In [ ]:
# CELL 3 — Install dependencies and verify GPU availability
# Purpose: ensure required packages are present and GPU is available for training.

!pip install -q librosa scikit-learn tqdm

import tensorflow as tf
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU Available:", bool(gpus))
if gpus:
    print("✅ GPU is enabled")
else:
    print("⚠️ No GPU detected. Use Runtime -> Change runtime type -> GPU if available.")


In [ ]:
# CELL 4 — Preprocessing function (matches training exactly)
# Purpose: given any WAV file, produce a (1,128,128,3) tensor identical to training pipeline,
# with robust audio loading and basic checks.

import librosa
import numpy as np
from scipy.ndimage import zoom
import soundfile as sf

def preprocess_audio_for_inference(
    file_path,
    sample_rate=16000,
    duration=2.0,
    n_fft=2048,
    hop_length=512,
    n_mels=128
):
    """
    Load a WAV (or other audio) file robustly, force fixed duration
    and return (1,128,128,3) float32 tensor identical to train preprocess.
    Raises a descriptive exception if loading fails.
    """
    target_length = int(sample_rate * duration)

    # Robust load: first try soundfile (preserves native), else fallback to librosa
    try:
        audio, sr = sf.read(file_path)
        # If stereo, convert to mono
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)
        # Resample if needed
        if sr != sample_rate:
            audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=sample_rate)
            sr = sample_rate
    except Exception as e_sf:
        # fallback to librosa.load (safer cross-format)
        try:
            audio, sr = librosa.load(file_path, sr=sample_rate)
        except Exception as e_lb:
            raise RuntimeError(f"Failed to load audio with soundfile ({e_sf}) and librosa ({e_lb}).")

    # Ensure float32
    audio = audio.astype(np.float32)

    # Trim / pad to exact duration
    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)), mode="constant")
    else:
        audio = audio[:target_length]

    # Compute mel spectrogram and convert to dB (log)
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sample_rate,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Resize to 128x128 using scipy.ndimage.zoom (order=1 -> bilinear)
    z_row = 128 / mel_db.shape[0]
    z_col = 128 / mel_db.shape[1]
    mel_resized = zoom(mel_db, (z_row, z_col), order=1)

    # Channel 0 normalization (0-1)
    eps = 1e-8
    c0 = (mel_resized - mel_resized.min()) / (mel_resized.max() - mel_resized.min() + eps)

    # Delta and delta-delta on the original mel (before resize) for stability
    delta = librosa.feature.delta(mel_db)
    delta2 = librosa.feature.delta(mel_db, order=2)

    delta_resized = zoom(delta, (128/delta.shape[0], 128/delta.shape[1]), order=1)
    delta2_resized = zoom(delta2, (128/delta2.shape[0], 128/delta2.shape[1]), order=1)

    c1 = (delta_resized - delta_resized.min()) / (delta_resized.max() - delta_resized.min() + eps)
    c2 = (delta2_resized - delta2_resized.min()) / (delta2_resized.max() - delta2_resized.min() + eps)

    # Stack and ensure float32
    mel_3ch = np.stack([c0, c1, c2], axis=-1).astype(np.float32)

    # Final sanity check
    if mel_3ch.shape != (128, 128, 3):
        raise ValueError(f"Preprocessed shape is wrong: {mel_3ch.shape}")

    return np.expand_dims(mel_3ch, axis=0)  # (1,128,128,3)


In [ ]:
# CELL 5 — Load dataset from .npy files and sanitize
# Purpose: load all .npy mel arrays and drop/repair invalid ones before training.

import glob
import numpy as np
from tqdm import tqdm
import os

def load_and_sanitize_dataset(base_dir):
    """
    Load .npy files from base_dir/real and base_dir/fake.
    Validate shape, dtype, finite values and normalize.
    Returns X (N,128,128,3) and y (N,)
    """
    real_files = sorted(glob.glob(os.path.join(base_dir, "real", "*.npy")))
    fake_files = sorted(glob.glob(os.path.join(base_dir, "fake", "*.npy")))

    X_list = []
    y_list = []
    dropped = 0

    def sanitize(arr, fpath):
        # Convert to numpy array and check
        if not isinstance(arr, np.ndarray):
            raise ValueError("Loaded object is not numpy array.")
        # Accept either (128,128,3) or (128,128) (if single channel accidentally)
        if arr.shape == (128,128):
            # expand to 3 identical channels
            arr = np.stack([arr, arr, arr], axis=-1)
        if arr.shape != (128,128,3):
            raise ValueError(f"Unexpected shape: {arr.shape}")
        # replace NaN/Inf
        if not np.isfinite(arr).all():
            raise ValueError("Array contains NaN or Inf.")
        # Ensure float32
        arr = arr.astype(np.float32)
        # Per-channel normalization (0-1) to be safe
        for c in range(3):
            ch = arr[:,:,c]
            mn, mx = ch.min(), ch.max()
            if mx - mn > 1e-12:
                arr[:,:,c] = (ch - mn) / (mx - mn)
            else:
                arr[:,:,c] = 0.0
        return arr

    print(f"Found {len(real_files)} real, {len(fake_files)} fake files. Validating...")
    for f in tqdm(real_files, desc="Real"):
        try:
            arr = np.load(f)
            arr = sanitize(arr, f)
            X_list.append(arr)
            y_list.append(1)
        except Exception as e:
            dropped += 1
            print(f"  Dropped (real): {os.path.basename(f)} -> {e}")

    for f in tqdm(fake_files, desc="Fake"):
        try:
            arr = np.load(f)
            arr = sanitize(arr, f)
            X_list.append(arr)
            y_list.append(0)
        except Exception as e:
            dropped += 1
            print(f"  Dropped (fake): {os.path.basename(f)} -> {e}")

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    print(f"Kept: {len(X)} samples. Dropped: {dropped} samples.")
    return X, y

# Run the loader
X, y = load_and_sanitize_dataset(BASE)
print("Raw X shape after load:", X.shape)

# If dataset is empty or too small, raise an error early
if X.shape[0] < 10:
    raise RuntimeError("Too few valid samples after sanitization. Fix dataset or preprocessing.")


In [ ]:
# CELL 6 — Prepare shapes and train/val/test splits
# Purpose: ensure channel dimension exists and split dataset speaker-agnostically if possible.

from sklearn.model_selection import train_test_split

# Ensure 4D (N,128,128,3)
if X.ndim != 4:
    raise ValueError(f"Expected X ndim=4 but got {X.ndim}")

# Cast types
X = X.astype(np.float32)
y = y.astype(np.int32)

# Stratified split: 70% train, 20% val, 10% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=1/3, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


In [ ]:
# CELL 7 — Build and compile CNN model
# Purpose: define the model architecture and compile with metrics including AUC.

from tensorflow.keras import layers, models, optimizers
import tensorflow as tf

def build_cnn(input_shape):
    inp = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D((2,2))(x)

    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D((2,2))(x)

    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool2D((2,2))(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inp, out)
    model.compile(
        optimizer=optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model

input_shape = X_train.shape[1:]
model = build_cnn(input_shape)
model.summary()


In [ ]:
# CELL 8 — Callbacks and model training
# Purpose: use consistent metric for checkpoint & early stopping (val_auc),
# add ReduceLROnPlateau also monitoring val_auc for consistency.

import tensorflow as tf

BEST_PATH = "/content/drive/MyDrive/best_model.h5"
FINAL_PATH = "/content/drive/MyDrive/voiceguard_cnn.h5"

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    BEST_PATH,
    monitor="val_auc",     # use AUC to decide the best model consistently
    mode="max",
    save_best_only=True,
    verbose=1
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_auc", patience=3, factor=0.5, verbose=1, mode="max"
)
early = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc", patience=6, restore_best_weights=True, verbose=1, mode="max"
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[checkpoint, reduce_lr, early],
    verbose=1
)


In [ ]:
# CELL 9 — Save final model & history to Drive
# Purpose: save the final model (weights after training/early-stop) and a copy of the best checkpoint exists already.

model.save(FINAL_PATH)
print(f"✅ Saved final model to {FINAL_PATH}")
print(f"✅ Best checkpoint saved to {BEST_PATH} (by ModelCheckpoint during training)")


In [ ]:
# CELL 10 — Load the saved best_model.h5 explicitly and evaluate on test set
# Purpose: always evaluate the exact checkpoint picked by ModelCheckpoint

from tensorflow.keras.models import load_model
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import numpy as np

if os.path.exists(BEST_PATH):
    model_eval = load_model(BEST_PATH, compile=False)  # load saved checkpoint
    print("✅ Loaded best checkpoint for evaluation:", BEST_PATH)
else:
    # fallback - use in-memory model (shouldn't normally happen)
    print("⚠️ BEST_PATH not found — using in-memory model for evaluation.")
    model_eval = model

# Predictions
y_prob = model_eval.predict(X_test, batch_size=32).ravel()

# Basic checks
if len(np.unique(y_test)) == 1:
    print("⚠️ Test set contains only one class; ROC AUC cannot be computed.")
else:
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"🎯 Test ROC AUC = {auc_score:.4f}")

# Default threshold metrics
y_pred = (y_prob >= 0.5).astype(int)
print("\nConfusion Matrix (threshold=0.5):")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report (threshold=0.5):")
print(classification_report(y_test, y_pred, digits=4))

# Save predictions CSV for debugging
import pandas as pd
out_df = pd.DataFrame({
    "y_true": y_test,
    "y_prob": y_prob,
    "y_pred_0.5": y_pred
})
out_csv = "/content/drive/MyDrive/test_predictions.csv"
out_df.to_csv(out_csv, index=False)
print("✅ Saved test predictions to:", out_csv)


In [ ]:
# CELL 11 — Plot training curves and save image
# Purpose: visualize train/val accuracy, loss, and AUC over epochs.

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.history.get('accuracy', []), label='Train')
axes[0].plot(history.history.get('val_accuracy', []), label='Val')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history.get('loss', []), label='Train')
axes[1].plot(history.history.get('val_loss', []), label='Val')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(history.history.get('auc', []), label='Train')
axes[2].plot(history.history.get('val_auc', []), label='Val')
axes[2].set_title('AUC')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/training_history.png', dpi=150)
plt.show()
print("✅ Saved training_history.png to Google Drive")


In [ ]:
# CELL 12 — Inference function using the exact preprocessing
# Purpose: run prediction on any uploaded audio file using the same preprocessing.

from tensorflow.keras.models import load_model

# Load model for inference (best checkpoint)
MODEL_FOR_INFER = BEST_PATH  # using the same best checkpoint

# Load once (re-use across multiple predicts)
_model_infer = load_model(MODEL_FOR_INFER)

def predict_file(file_path, threshold=0.5):
    """
    Run preprocessing -> model -> probability and label
    Returns (probability, label)
    """
    X_in = preprocess_audio_for_inference(file_path)
    prob = float(_model_infer.predict(X_in, batch_size=1).ravel()[0])
    label = "REAL" if prob >= threshold else "FAKE"
    print(f"File: {file_path}  |  Prediction: {label}  |  Confidence: {prob:.4f}")
    return prob, label

# Example usage after uploading a file to Colab:
# predict_file("/content/my_audio.wav")
